# Automatic Block Synchronization Pipeline

This notebook automates the manual LED alignment and drift correction steps from the block synchronization workflow. It uses the same block setup and initial sync as the manual notebook, then:

1. **Block setup** — Same as manual: instantiate block, set channeldict, run data prep.
2. **Brightness** — Load existing or create with **auto ROI** (2×2 at brightest spot of first frame); falls back to manual ROI if auto fails.
3. **Initial sync** — `simple_sync_build(block)`.
4. **LED blink detection** — `find_led_blink_frames()` to get blink peaks for alignment.
5. **Automated first-blink alignment** — Align ON edge (LED turning on) with the lowest-brightness frame; apply `shift_eye_df_by_index`.
6. **Automated drift correction** — Insert/remove whole frames so each LED blink stays aligned to TTL over time.
7. **Verification** — Plot results for user check.
8. **Final merge and export** — Same as manual pipeline.

In [1]:
# ============================================================================
# IMPORTS
# ============================================================================
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

from bokeh.io import output_notebook, show
output_notebook()

from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing.block_sync_core import (
    simple_sync_build,
    shift_eye_df_by_index,
    compute_led_alignment_shift,
    apply_drift_correction,
    build_final_sync_df_merge_nearest,
    verify_final_df_against_sources,
    export_final_sync_df,
    find_jittery_frames,
    export_eye_data_2d,
    describe_eye_tick,
)
from eye_tracking_system_tools.preprocessing.block_sync_visualization import (
    plot_simple_sync_bokeh,
    plot_led_off_events_viewer,
    interactive_sync_tool_bokeh,
    plot_sync_verification_verbose_bokeh,
)

Loading BokehJS ...

## Step 1: Block instantiation

Same as the manual notebook: set paths, block numbers, animal, and channeldict.

In [2]:
bad_blocks = []
experiment_path = Path(r"D:\sample_data_for_eye_repo")  # adjust to your path
block_numbers = [15]
animal = 'PV_106'

block_collection = uf.block_generator(
    block_numbers=block_numbers,
    experiment_path=experiment_path,
    animal=animal,
    bad_blocks=bad_blocks,
)
for block in block_collection:
    block.channeldict = None
    if block.animal_call == 'PV_208':
        block.channeldict = {1: 'LED_driver', 7: 'L_eye_TTL', 2: 'Arena_TTL', 8: 'R_eye_TTL'}
    elif block.animal_call == "TE_21":
        block.channeldict = {1: 'Arena_TTL', 4: 'LED_driver', 5: 'R_eye_TTL', 8: 'L_eye_TTL'}
    elif block.animal_call == "PV_106":
        block.channeldict = {1: 'LED_driver', 7: 'L_eye_TTL', 2: 'Arena_TTL', 8: 'R_eye_TTL'}
    elif block.animal_call == "PV_126":
        block.channeldict = {1: 'LED_driver', 7: 'L_eye_TTL', 2: 'Arena_TTL', 8: 'R_eye_TTL'}

block_dict = {str(b.block_num): b for b in block_collection}
block = list(block_collection)[0]  # single block for this run

instantiated block number 015 at Path: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015, new OE version
Found the sample rate for block 015 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\oe_files\PV106_IMU_trial4_prey_2025-09-04_13-24-17\Record Node 106...
Analog channel numbers contain duplicates!!! Reordering numbers serially.

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode, no metadata file)
retrieving zertoh sample number for block 015
got it!


## Step 2: Data preparation

Handle videos, parse Open Ephys events, handle arena. For brightness: if the pkl already exists it is loaded; otherwise brightness is created using **auto ROI** (2×2 at brightest spot). Set `use_auto_roi=False` to force manual ROI selection.

In [3]:
block.handle_eye_videos()
block.parse_open_ephys_events()
block.handle_arena_files()

# Use auto ROI when creating brightness (falls back to manual if auto fails)
use_auto_roi = True
block.get_eye_brightness_vectors(threshold_value=30, export=True, use_auto_roi=use_auto_roi)

handling eye video files
converting videos...
converting files: ['D:\\sample_data_for_eye_repo\\PV_106\\2025_09_04\\block_015\\eye_videos\\LE\\imu_trial4_prey\\imu_trial4_prey.h264', 'D:\\sample_data_for_eye_repo\\PV_106\\2025_09_04\\block_015\\eye_videos\\RE\\imu_trial4_prey\\imu_trial4_prey.h264'] 
 avoiding conversion on files: ['D:\\sample_data_for_eye_repo\\PV_106\\2025_09_04\\block_015\\eye_videos\\LE\\imu_trial4_prey\\imu_trial4_prey_LE.mp4', 'D:\\sample_data_for_eye_repo\\PV_106\\2025_09_04\\block_015\\eye_videos\\RE\\imu_trial4_prey\\imu_trial4_prey.mp4']
The file D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\eye_videos\RE\imu_trial4_prey\imu_trial4_prey.mp4 already exists, no conversion necessary
Validating videos...
The video named imu_trial4_prey_LE.mp4 has reported 19800 frames and has 19800 frames, it has dropped 0 frames
The video named imu_trial4_prey.mp4 has reported 19701 frames and has 19701 frames, it has dropped 0 frames
running parse_open_ephys_events...

## Step 3: Initial sync

Build per-eye DataFrames aligned to OE time using the first TTL anchor.

In [4]:
dfL, dfR = simple_sync_build(block, export=True)
print("Left tick ≈ %.3f ms" % describe_eye_tick(dfL))
print("Right tick ≈ %.3f ms" % describe_eye_tick(dfR))

[LEFT] frames=19,800 | median fps=62.496 | CoV(dt)=6.88% | outliers(±1–99%)=1.90%
[WARN] LEFT: CoV(dt) > 5.0%. Stream may be unstable.
[RIGHT] frames=19,701 | median fps=62.496 | CoV(dt)=34.92% | outliers(±1–99%)=1.96%
[WARN] RIGHT: CoV(dt) > 5.0%. Stream may be unstable.
[OK] Saved: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\analysis\eye_left_simple_sync.csv
[OK] Saved: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\analysis\eye_right_simple_sync.csv
Left tick ≈ 16.000 ms
Right tick ≈ 16.000 ms


## Step 4: LED blink detection

Detect LED blink frames (for removal) and **peak** frames (center of each blink, for alignment). Sets `led_blink_frames_l/r` and `led_blink_peak_frames_l/r`.

In [5]:
block.find_led_blink_frames(plot=False)
print(f"Left: {len(block.led_blink_peak_frames_l)} blink peaks; Right: {len(block.led_blink_peak_frames_r)} blink peaks")

hi new version
collecting left-eye data
data length is 19800


100%|██████████| 13/13 [00:00<00:00, 4333.65it/s]


z_score length is 19800
collecting right eye data
data length is 19701


100%|██████████| 13/13 [00:00<00:00, 4336.06it/s]

z_score length is 19701
Left: 6 blink peaks; Right: 6 blink peaks


## Step 5: Automated first-blink alignment

Compute shift so the first blink's lowest-brightness frame aligns with the first ON edge (LED turning on). ON/OFF edges are deduced from intervals (34 ms OFF-ON, ~60 s to next blink).

In [6]:
shift_left, shift_right = compute_led_alignment_shift(block, dfL, dfR)
print(f"Computed shifts: left={shift_left}, right={shift_right}")

dfL_shifted = shift_eye_df_by_index(dfL, shift_left)
dfR_shifted = shift_eye_df_by_index(dfR, shift_right)

Computed shifts: left=0, right=0


In [ ]:
# Verbose verification: where the analysis places each "current" row and expected peak, and what correction it would apply
# Green solid = LED ON (target). Cyan dashed = L "current" row at each event. Orange dashed = R "current". Light blue/coral dotted = where expected peak frame lands.
layout_verbose = plot_sync_verification_verbose_bokeh(block, dfL_corrected, dfR_corrected, drift_threshold_frames=1.0)
show(layout_verbose)

## Step 6: Automated drift correction

Correct cumulative drift by inserting or removing whole frames so each LED blink stays aligned to the corresponding ON TTL. Whole-frame steps only.

In [7]:
dfL_corrected, dfR_corrected = apply_drift_correction(
    block,
    dfL_shifted,
    dfR_shifted,
    max_event_offset_frac=0.6,
    min_interval_frames=1500,
    verbose=True,
)

[INFO] Drift correction LEFT (vs LED): inserted 5, removed 0 frame(s)
[INFO] Drift correction RIGHT (vs LED): inserted 5, removed 0 frame(s)


## Step 7: Verification

Plot brightness vs OE time with LED verticals to confirm alignment. Optionally use the interactive sync tool or LED-off events viewer for detailed check.

In [8]:
plot_simple_sync_bokeh(block, dfL_corrected, dfR_corrected, show_led=True)

[INFO] Slider tick ≈ 16.000 ms (Left), 16.000 ms (Right)


In [ ]:
# Optional: interactive tool to fine-tune or inspect
# result_sync = {}
# display(interactive_sync_tool_bokeh(block, dfL_corrected, dfR_corrected, show_led=True, result_container=result_sync))
# If you export from the tool, use result_sync['dfL'] and result_sync['dfR'] in the next step.

# Optional: LED-off events viewer (requires block.oe_rec and block.final_sync_df)
# plot_led_off_events_viewer(block, window_ms=100)

## Step 8: Final merge and export

Merge corrected eye data onto the 60 Hz arena grid and export. Then run jitter correction and LED blink removal as in the manual pipeline if needed.

In [ ]:
final_df = build_final_sync_df_merge_nearest(
    block,
    dfL_corrected,
    dfR_corrected,
    pre_shift_left=0,
    pre_shift_right=0,
    export_csv=True,
    csv_name="blocksync_df.csv",
    verbose=True,
)
block.final_sync_df = final_df

In [ ]:
# Optional: verify final df and run jitter / LED blink removal (same as manual notebook)
# verify_final_df_against_sources(block)
# block.find_led_blink_frames(plot=True)
# block.remove_led_blinks_from_eye_df()  # then re-export if needed
# find_jittery_frames(block)
# export_eye_data_2d(block, ...)